# L2-04 配套 notebook：简单基线与效应迁移族

对应课文 [`docs/lessons/L2-04-简单基线与效应迁移族.md`](../docs/lessons/L2-04-简单基线与效应迁移族.md)。

本 notebook 用**纯 NumPy** 重建 B0 / B1 / B2 / B3 四级统计基线的估计量与评估骨架。
全部用合成小矩阵，**不加载任何真实扰动面板**，因此这里的数字**不能**当作本赛、
公开面板或 H1 基准上的实测结果。

**它回答四个问题：**

| 单元 | 问题 |
|---|---|
| 1 | B0 与 B1 到底怎么算？和课文 §7.1 的手算例子逐位对齐吗？ |
| 2 | 为什么跨背景聚合用中位数而不是均值？差多少？ |
| 3 | LOCO 到底留出什么？B1 的"查找表"性质意味着什么？ |
| 4 | B2 的相似度加权是怎么把权重集中到近邻背景上的？ |
| 5 | 怎么把 B1 的"是否真的优于 B0"接到噪声地板上？ |

**前置阅读**：[L1-02 证据基线与噪声地板](L1-02-证据基线与噪声地板.md)（地板的含义与数字）、
[L2-03 基础模型路线](L2-03-基础模型路线.md) §6.4（`beta_frac` 分档）。

> **证据分级**：本 notebook 的全部输出都是**工程假设**（本项目自己构造的演示）。
> 唯一一个真实数字是官方 H1 基准报告的 B0 参考值 `-0.045230687652238`，
> 它在单元 5 只作**引用**出现，本 notebook 没有复跑它。

## 环境与版本

只用 NumPy 与 matplotlib。固定随机种子，保证每次运行结果一致。

In [ ]:
import sys
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

# 中文字体探测：没有就用英文标签，避免出现方块
for _f in ("Microsoft YaHei", "SimHei", "Noto Sans CJK SC", "WenQuanYi Zen Hei"):
    if any(_f.lower() in f.lower() for f in matplotlib.font_manager.get_font_names()):
        matplotlib.rcParams["font.sans-serif"] = [_f]
        _CJK = _f
        break
else:
    _CJK = None
matplotlib.rcParams["axes.unicode_minus"] = False

print("python     =", sys.version.split()[0])
print("numpy      =", np.__version__)
print("matplotlib =", matplotlib.__version__)
print("CJK 字体    =", _CJK if _CJK else "未找到（图中标签用英文）")

rng = np.random.default_rng(20260918)
print("rng seed   = 20260918")

## 单元 1：手算 B0 与 B1

复现课文 §8.1 的 4 基因例子，逐位对齐。

- **B0**：$\widehat{\mu}^{B0}_{c,t,g} = \mu^{\mathrm{NTC}}_{c,g}$ —— 不含 $t$，对所有靶点输出相同。
- **B1**：$\widehat{\mu}^{B1}_{c,t,g} = \mu^{\mathrm{NTC}}_{c,g} + \overline{\Delta}_{t,g}$，
  其中 $\overline{\Delta}_{t,g} = \operatorname{median}_{d\in\mathcal D_t}\Delta_{d,t,g}$，
  且 $\Delta_{d,t,g}$ **先在每个背景内部算**。

In [ ]:
GENES = ["G0", "G1", "G2", "G3"]

# 课文 §8.1 的四个数字，逐字抄过来
mu_ntc_d1 = np.array([0.10, 2.00, 0.50, 1.00])
pert_d1_t = np.array([0.30, 1.20, 0.50, 1.00])

mu_ntc_d2 = np.array([0.20, 1.00, 0.30, 2.00])
pert_d2_t = np.array([0.50, 0.40, 0.30, 2.00])

# 目标背景 c（第三次出现的那个背景，课文里 mu_ntc_c）
mu_ntc_c = np.array([0.15, 1.50, 0.40, 1.60])


def show(name, vec):
    print("%-28s %s" % (name, np.array2string(np.asarray(vec), precision=4)))


print("=== 每个背景内部先算 delta（绝不先混细胞）===")
delta_d1 = pert_d1_t - mu_ntc_d1
delta_d2 = pert_d2_t - mu_ntc_d2
show("Delta_d1", delta_d1)
show("Delta_d2", delta_d2)

print()
print("=== B1: 跨背景逐基因取中位数 ===")
# D_t = {d1, d2}：d3 没测这个靶点，所以不进聚合
D_t = np.vstack([delta_d1, delta_d2])
delta_bar = np.median(D_t, axis=0)
show("Delta_bar (median)", delta_bar)
print("课文 §8.1 给出   [0.25 -0.70  0.00  0.00]")

print()
print("=== 两个基线的预测 ===")
pred_B0 = mu_ntc_c.copy()
pred_B1 = mu_ntc_c + delta_bar
show("B0 = mu_ntc_c", pred_B0)
show("B1 = mu_ntc_c + dbar", pred_B1)
print("课文 §8.1 给出 B1 [0.40 0.80 0.40 1.60]")

assert np.allclose(delta_bar, [0.25, -0.70, 0.0, 0.0]), "delta_bar 与课文不一致"
assert np.allclose(pred_B1, [0.40, 0.80, 0.40, 1.60]), "B1 预测与课文不一致"
assert not np.allclose(pred_B0, pred_B1), "B1 必须与 B0 不同"
print()
print("OK: 与课文 §8.1 逐位一致")

print()
print("=== 关键性质：B0 完全不依赖靶点 t ===")
targets = ["ABCD1", "ACLY", "ADNP"]
B0_all = np.vstack([mu_ntc_c] * len(targets))
print("三个不同靶点的 B0 预测是否逐位相同:", bool((B0_all[0] == B0_all[1:] ).all()))
print("=> B0 对全部 300 个靶点输出同一个东西，所以它的身份 PDS 原始值必然是 0.5（并列）")

## 单元 2：中位数 vs 均值——一个异常背景的影响

课文 §3.2 说跨背景聚合要用中位数，理由是**稳健**。
这个单元量化它：让 5 个背景里出 1 个「测崩了」的背景，看两种聚合方式差多少。

这就是 min-max 与 median 的真实分野：均值会被一个离群背景拖走，中位数不会。

In [ ]:
# 5 个背景对同一个靶点、同一个基因的 delta
good = np.array([0.20, 0.25, 0.22, 0.18])      # 4 个正常背景
bad = np.array([6.00])                          # 1 个异常背景（批次炸了 / guide 效率异常）

whole = np.concatenate([good, bad])
print("4 个正常背景的 delta:", good)
print("1 个异常背景的 delta:", bad)
print()
print("跨背景均值     mean  = %.4f" % whole.mean())
print("跨背景中位数   median= %.4f" % np.median(whole))
print("只用正常背景的均值      = %.4f" % good.mean())
print()
print("异常背景把 mean 拉高了 %.4f（= %.1f 倍），median 几乎不动。"
      % (whole.mean() - np.median(whole), whole.mean() / np.median(whole)))
print()
print("=> 这正是课文 §3.2 选择 median 的理由：一个背景测崩了不该把整个 delta 拖歪。")
print("   注意：median 的稳健性需要至少 3 个背景才体现；2 个背景时 median == mean（课文 §8.1 的退化情形）。")

for n in (2, 3, 5, 9):
    sub = np.concatenate([good[:n - 1], bad]) if n > 1 else bad
    print("  背景数=%d  mean=%.4f  median=%.4f  相同=%s"
          % (n, sub.mean(), np.median(sub), np.isclose(sub.mean(), np.median(sub))))

## 单元 3：LOCO 协议骨架

**LOCO = leave-one-context-out**：留出一个细胞系当目标背景 $c$，用其余的背景当捐献背景 $\mathcal D$。

这个单元用 4 个合成"背景"演示协议，并暴露 B1 最关键的性质——
**$\overline{\Delta}_t$ 是所有目标背景共享的同一个向量**。

`delta_table` 就是 B1 的「全部知识」：一张 `(靶点, 基因)` 的表。
课文 §3.5 说 B1 有 0 个可训练参数，指的就是这张表是统计量而不是拟合参数。

In [ ]:
CONTEXTS = ["K562", "RPE1", "Jurkat", "HepG2"]   # 借公开面板的真实细胞系名
N_GENES = 6
TARGETS = ["ABCD1", "ACLY", "ADNP", "BAG1"]

# 构造一个合成 delta 表: (背景, 靶点, 基因)
base_effect = rng.normal(0.0, 0.30, size=(len(TARGETS), N_GENES))
delta = np.zeros((len(CONTEXTS), len(TARGETS), N_GENES))
for ci in range(len(CONTEXTS)):
    delta[ci] = base_effect + rng.normal(0.0, 0.10, size=(len(TARGETS), N_GENES))
# 让 HepG2 缺失第 0 号靶点 -> 触发 D_t 覆盖不完整
missing = (CONTEXTS.index("HepG2"), 0)

print("delta tensor shape = (背景, 靶点, 基因) =", delta.shape)
print("缺失记录:", CONTEXTS[missing[0]], "/", TARGETS[missing[1]])
print()


def loco_fold(held_out):
    """留出 held_out 背景，其余背景算 delta_bar。返回 (delta_bar, coverage)。"""
    donors = [c for c in CONTEXTS if c != held_out]
    bar = np.zeros((len(TARGETS), N_GENES))
    cov = np.zeros(len(TARGETS), dtype=int)
    for ti, t in enumerate(TARGETS):
        rows = []
        for dc in donors:
            di = CONTEXTS.index(dc)
            if (di, ti) == missing:
                continue          # 该背景没测这个靶点 -> 不进聚合
            rows.append(delta[di, ti])
        cov[ti] = len(rows)
        bar[ti] = np.median(np.vstack(rows), axis=0) if rows else np.nan
    return bar, cov


for held in CONTEXTS:
    bar, cov = loco_fold(held)
    print("留出 %-7s -> 捐献背景 %-34s 覆盖度 %s"
          % (held, str([c for c in CONTEXTS if c != held]), cov.tolist()))

print()
bar_k562, cov_k562 = loco_fold("K562")
bar_rpe1, cov_rpe1 = loco_fold("RPE1")
print("=== B1 的核心性质：delta 表换个目标背景会不会变？ ===")
print("留出 K562 得到的 delta_bar 第 0 行:", np.round(bar_k562[0], 4))
print("留出 RPE1 得到的 delta_bar 第 0 行:", np.round(bar_rpe1[0], 4))
print("两者不同（因为捐献背景集合不同），但每个 delta_bar 内部对所有目标背景都是同一张表。")
print()
print("=== 零覆盖必须回退 B0 ===")
print("靶点 %s 在留出 HepG2 时有 %d 个背景覆盖；若某靶点覆盖度为 0，"
      % (TARGETS[0], cov_k562[0]))
# np.median([]) 会返回 nan 并打一条 RuntimeWarning。这里显式处理，因为
# "覆盖度为 0 时该怎么办" 正是本单元要讲的判断，不该让 NumPy 的警告替我们回答。
with np.errstate(invalid="ignore"):
    import warnings
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        _empty = np.median(np.array([]))
print("median(空) = %r -> 得到 nan 而不是 0；正确做法是显式回退 B0，" % _empty)
print("   而不是拿名字相似的基因顶替，也不是把 nan 当零效应继续往下算。")

## 单元 4：B2 把权重集中到哪里？

B2 给更相似的背景更大权重：

$$w_{c,d}=\frac{\exp(s(z_c,z_d)/\tau)}{\sum_{d'}\exp(s(z_c,z_{d'})/\tau)}$$

这个单元构造一个**近邻背景对**（共享技术指纹，NTC 因此更相似）和一个**远处背景**，
展示 B2 的权重大量流向近邻——这正是课文 §9.3 那道题的机制。

In [ ]:
z_ctx = {"K562":   np.array([1.00, 0.90, 0.10, 0.05]),   # 与 RPE1 同平台
         "RPE1":   np.array([0.98, 0.88, 0.12, 0.06]),
         "Jurkat": np.array([0.10, 0.05, 1.00, 0.92]),   # 跨平台，远处
         "HepG2":  np.array([0.12, 0.04, 0.95, 0.90])}


def softmax(x):
    e = np.exp(x - x.max())
    return e / e.sum()


def b2_weights(target_ctx, tau):
    donors = [c for c in CONTEXTS if c != target_ctx]
    zc = z_ctx[target_ctx]
    s = np.array([float(zc @ z_ctx[d] / (np.linalg.norm(zc) * np.linalg.norm(z_ctx[d])))
                  for d in donors])
    return donors, s, softmax(s / tau)


for tau in (0.02, 0.10):
    print("=== 目标背景 Jurkat（跨平台）, tau = %.2f ===" % tau)
    donors, s, w = b2_weights("Jurkat", tau)
    for d, si, wi in zip(donors, s, w):
        print("   %-7s cosine=%.4f  weight=%.6f" % (d, si, wi))
    print("   最大权重 = %.6f（其余背景被压到几乎为零）" % w.max())
    print()

print("=== 目标背景 K562（有同平台近邻）, tau = 0.02 ===")
donors, s, w = b2_weights("K562", 0.02)
for d, si, wi in zip(donors, s, w):
    print("   %-7s cosine=%.4f  weight=%.6f" % (d, si, wi))
print("   最大权重 = %.6f（K562 的近邻是 RPE1）" % w.max())
print()

print("=> tau 越小，权重越向最相似的背景集中。")
print("=> Jurkat 的最近邻是 HepG2（同为'远处簇'），K562 的最近邻是 RPE1。")
print("   B2 能不能赢，完全取决于'这个近邻的扰动响应是否也相似'。")
print("   课文 §9.3 的反例：同平台近邻让 B2 赢 0.003-0.004，")
print("   跨平台留出让 B2 输 0.054 —— 平均分把失败盖住了。")

## 单元 5：B1 是否真的优于 B0——接到噪声地板

这是本课最重要的一步。课文 §9.2 的结论是：
**光有「B1 比 B0 高 0.0054」不足以宣布改进**，还缺同协议下的重复与 `avg_score` 尺度的地板。

这个单元把这条纪律写成可执行的形式：给定一个"分数差"和"地板"，做机械判定。

In [ ]:
# 真实引用值：官方 H1 开发基准报告的 unchanged-control baseline（= B0）
# 来源 references/vcc2026-h1-benchmark/README.md，本 notebook 未复跑。
B0_OFFICIAL = -0.045230687652238

# L1-02 §5.4 本机实测的噪声地板（群体表达谱平方距离，每背景 400 细胞）
FLOOR_SQ_DIST = {"A": 28.8, "B": 29.7, "C": 32.9}
FLOOR_THRESHOLD = {"A": 31.4, "B": 33.9, "C": 38.6}

print("官方 H1 基准报告的 B0 参考值 avg_score = %.15f" % B0_OFFICIAL)
print("  ^ 这是官方基准 README 的数，本 notebook 没有复跑它（需 15.48 GB 下载）。")
print()
print("L1-02 §5.4 本机实测噪声地板（平方距离）:")
for k in FLOOR_SQ_DIST:
    print("   背景 %s: 地板 %.1f, 置换 99%% 判定阈值 %.1f" % (k, FLOOR_SQ_DIST[k], FLOOR_THRESHOLD[k]))
print()


def judge(delta_score, floor, n_repeats=0, same_protocol=True):
    """把'是否算改进'写成机械判定, 而不是感觉。"""
    reasons = []
    if not same_protocol:
        reasons.append("两个分数不在同一协议/面板下得到，不能直接相减")
    if n_repeats < 3:
        reasons.append("缺少重复测量（建议 >= 3 个种子/划分），区间未知")
    if abs(delta_score) <= floor:
        reasons.append("分数差 %.6f 未超过地板 %.6f" % (delta_score, floor))
    if reasons:
        return False, reasons
    return True, ["超过地板且同协议下有重复"]


print("=== 情形 1：课文 §9.2 给的那组数字 ===")
ok, why = judge(delta_score=0.0054, floor=0.0010, n_repeats=0, same_protocol=False)
print("B0=-0.0452  B1=-0.0398  差=+0.0054")
print("判定:", "算改进" if ok else "不算改进")
for r in why:
    print("   -", r)
print()

print("=== 情形 2：同协议、有重复、超过地板 ===")
ok, why = judge(delta_score=0.0054, floor=0.0010, n_repeats=5, same_protocol=True)
print("判定:", "算改进" if ok else "不算改进", "|", "; ".join(why))
print()

print("=== 情形 3：同协议有重复，但差值小于地板 ===")
ok, why = judge(delta_score=0.0003, floor=0.0010, n_repeats=5, same_protocol=True)
print("判定:", "算改进" if ok else "不算改进")
for r in why:
    print("   -", r)
print()

print("=== 为什么不能把 L1-02 的地板直接用到 avg_score 上 ===")
print("L1-02 的地板单位是【群体表达谱平方距离】，背景 A 是 28.8。")
print("avg_score 是六项缩放分的等权平均，单位完全不同。")
print("六项里 DE 类指标要过 Wilcoxon + BH 校正，噪声被非线性放大。")
print("=> 要么在地板所在的同一个量上比较，要么另行测量 avg_score 尺度的地板。")
print("   L1-02 §7 与 §10 已明确列出这条不可外推的边界。")

## 单元 6：读图——机会窗口

把 [L2-03](L2-03-基础模型路线.md) §6.4 与 notebook 08 的机会窗口曲线放在一起看：
它给出**什么时候才值得从 B1 升级到高容量模型**。

图只画机制，用合成数据；分档阈值 0.70 / 0.50 是 [L2-03](L2-03-基础模型路线.md) 的工程假设。

In [ ]:
# 来自 notebook 08 单元 5 的实测点（B1 / 完美知道 Gamma 的上界）
GAMMA = np.array([0.05, 0.10, 0.20, 0.40, 0.80, 1.60, 3.20])
RATIO = np.array([0.9985, 0.9940, 0.9764, 0.9126, 0.7305, 0.4152, 0.1496])

fig, axes = plt.subplots(1, 2, figsize=(11.0, 4.0), dpi=110)

ax = axes[0]
ax.semilogx(GAMMA, RATIO, "o-", color="#c0392b", lw=1.6, ms=6)
ax.set_xlabel("背景特异分量 Γ 的强度（对数刻度）" if _CJK else "Gamma strength (log)")
ax.set_ylabel("B1 / upper bound" if not _CJK else "B1 / 上界")
ax.set_title("机会窗口：Γ 越强，B1 越吃亏" if _CJK else "Opportunity window",
             fontsize=11)
ax.grid(alpha=0.3, ls="--")
ax.set_ylim(0, 1.06)
for x, y in zip(GAMMA, RATIO):
    ax.annotate("%.3f" % y, (x, y), textcoords="offset points",
                xytext=(0, -14), ha="center", fontsize=8, color="#8a2b2b")

ax2 = axes[1]
beta_frac = np.linspace(0.0, 1.0, 200)
# 分档用 L2-03 的阈值 0.70 / 0.50（工程假设）
color = np.where(beta_frac >= 0.70, "#82b366",
                 np.where(beta_frac >= 0.50, "#d6b656", "#b85450"))
ax2.scatter(beta_frac, np.zeros_like(beta_frac), c=color, s=14, marker="s")
ax2.set_xlim(0, 1)
ax2.set_ylim(-0.5, 0.5)
ax2.set_yticks([])
ax2.set_xlabel("beta_frac = Var(beta) / [Var(beta) + mean Var(Gamma)]"
               if not _CJK else "beta_frac（β 主导 ← → Γ 主导）")
ax2.set_title("靶点分档：算力只投低档" if _CJK else "Target bucketing", fontsize=11)
ax2.axvline(0.50, color="#666666", ls="--", lw=1)
ax2.axvline(0.70, color="#666666", ls="--", lw=1)
ax2.text(0.85, 0.25, "B1 就够" if _CJK else "B1 enough", ha="center", fontsize=9, color="#2f6b2f")
ax2.text(0.60, 0.25, "先 B1" if _CJK else "B1 first", ha="center", fontsize=9, color="#8a6d1f")
ax2.text(0.25, 0.25, "投模型" if _CJK else "spend here", ha="center", fontsize=9, color="#8a2b2b")
ax2.grid(alpha=0.3, axis="x", ls="--")

fig.tight_layout()
from pathlib import Path
_out = Path("../output/tmp")
_out.mkdir(parents=True, exist_ok=True)
fig.savefig(_out / "nb09-b1-opportunity-window.png", bbox_inches="tight")
plt.close(fig)
print("wrote", _out / "nb09-b1-opportunity-window.png", "(Git 忽略目录)")
print()
print("        Γ 强度   B1/上界")
for g, r in zip(GAMMA, RATIO):
    print("   %8.2f   %6.4f" % (g, r))
print()
print("Γ 从 0.05 到 3.20，比值从 0.9985 掉到 0.1496。")
print("=> Γ 弱时 B1 几乎等于上界（没什么可学的）；Γ 强时 B1 才留出空间。")
print("=> 但注意：Γ 强也意味着噪声大、每个背景都得单独测 —— 这正是稀缺的东西。")

## 自检清单

跑完本 notebook，你应当能回答：

1. **B0 与 B1 的预测差在哪里？** —— B0 不含 $t$，对所有靶点输出相同；B1 加了跨背景中位数 delta。
2. **B1 有几个可训练参数？** —— **0 个**。$\overline{\Delta}_t$ 是统计量，不是拟合参数；B1 是一张查找表。
3. **为什么聚合用 median 不用 mean？** —— 一个背景测崩了不该把整个 delta 拖歪（单元 2 量化）。
4. **$\mathcal D_t = \varnothing$ 时怎么办？** —— 回退 B0，不能拿名字相似的基因顶替。
5. **B2 的收益可能来自哪里？** —— 可能是技术相似性而非生物相似性；要用批次/平台标签做负对照（课文 §9.3）。
6. **「B1 比 B0 高 0.0054」够不够宣布改进？** —— 不够。缺同协议、缺重复、缺 `avg_score` 尺度的地板。

---

### 三件本 notebook **没有**做的事

1. **没有跑任何真实的 B1–B3 分数。** 全部是合成矩阵。唯一真实数字是
   `B0_OFFICIAL = -0.045230687652238`，来自官方 H1 基准 README，**未在本机复跑**。
2. **没有下载 15.48 GB 的 H1 数据集**，也没有下载公开扰动面板（`datasets.yaml` 的 Replogle / Nadig / Norman）。
3. **没有把预测接到计数生成器。** 输出停在群体均值空间，末端流程见 [L3-03](L3-03-单细胞原始计数生成.md)。

In [ ]:
print("ALL CELLS OK")